# Day 89: A cleaning report anyone can read

**Goal:** put everything I know about the Ghana soil table on one page, so a teacher, a boss or a professor can read it in two minutes.

**The picture:** a school report card. Not every test paper, just one table: *check, result, what I did about it*.

> Restart the kernel, then run top to bottom.

## 0. Setup
The functions live two folders up, so I add that folder to the search path **before** importing.

In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
from agronomic_functions import build_flat, quality_flags, dataset_alarm, cleaning_report
from db_config import run_query

## 1. Rebuild the table (Days 84 to 87)

In [ ]:
flat = build_flat("orgc", ["ph", "clay", "sand"])
checked = quality_flags(flat)
meta = run_query("SELECT layer_id, dataset_id FROM wosis_orgc WHERE country_name = 'Ghana'")
traced = pd.merge(checked, meta, on="layer_id", how="left")
print(len(traced))        # must be 1756

## 2. The lesson: three small ideas
1. **A report is a short table, one row per check.** Each row says what I checked, what I found and what I decided.
2. **Build a table from a list of rows.** Each row is a small dictionary (name: value). A list of them becomes a DataFrame, and the keys become the column names.
3. **The numbers come from code, never from typing.** If the data changes tomorrow, a computed number updates itself. A typed one quietly becomes a lie.

In [ ]:
rows = [
    {"check": "Duplicate rows", "result": 0,  "action": "none needed"},
    {"check": "Missing pH",     "result": 14, "action": "keep, leave empty"},
]
print(pd.DataFrame(rows))

## 3. Exercise on made-up data
**Prediction before running:** duplicated ids = 1, missing pH = 1, pH above 14 = 1.

An `action` must say what I do and why. "action" as an action is a placeholder and tells the reader nothing. A good action starts with a verb: *flag; impossible value, check the source*.

In [ ]:
toy = pd.DataFrame({
    "id": [1, 2, 2, 4],
    "ph": [6.1, None, 7.0, 15.0]
})

rows = [
    {"check": "Duplicate ids",
     "result": int(toy["id"].duplicated().sum()),
     "action": "investigate; same id, different values"},
    {"check": "Missing pH",
     "result": int(toy["ph"].isna().sum()),
     "action": "keep, leave empty"},
    {"check": "pH above 14",
     "result": int((toy["ph"] > 14).sum()),
     "action": "flag; impossible value, check the source"},
]

report = pd.DataFrame(rows)
print(report)
report.to_csv("toy_report.csv", index=False)

**Check question:** why must the numbers in a report come from code?
**My answer:** a typed number does not change when the data changes, so the report becomes wrong without anyone noticing. Computed numbers update when I rerun the cell, and there are fewer typing mistakes.

## 4. The real report
One row per check. Every `result` is computed. For missing values, the action also shows the percentage, computed in code.

> Your own code from the day may look a little different. The numbers must match.

In [ ]:
n_rows = len(traced)
dup_rows = int(traced.duplicated().sum())
dup_ids = int(traced["layer_id"].duplicated().sum())

hard_cols = ["flag_ph", "flag_clay_sand", "flag_clay", "flag_sand", "flag_oc_neg"]
hard_flags = int(traced[hard_cols].sum().sum())
soft_flags = int(traced["flag_oc_high"].sum())

alarm_names = ", ".join(dataset_alarm(traced, "orgc").index)

rows = [
    {"check": "Total layers", "result": n_rows, "action": "baseline count"},
    {"check": "Duplicate rows", "result": dup_rows,
     "action": "none needed" if dup_rows == 0 else "investigate; exact copies"},
    {"check": "Duplicate layer_id", "result": dup_ids,
     "action": "none needed" if dup_ids == 0 else "investigate; same id on different rows"},
    {"check": "Impossible values (hard flags)", "result": hard_flags,
     "action": "none needed" if hard_flags == 0 else "investigate; check the source"},
    {"check": "Unusual high carbon (soft flag)", "result": soft_flags,
     "action": "keep flags, investigate GH-GhaSP scale"},
    {"check": "Datasets that fail the alarm for orgc", "result": alarm_names,
     "action": "check dataset documentation for units"},
]

for col in ["ph", "clay", "sand"]:
    missing = int(traced[col].isna().sum())
    present = int(traced[col].notna().sum())
    assert missing + present == n_rows          # self-check: missing + present = all rows
    pct = missing / n_rows * 100
    if pct == 0:
        action = "none needed"
    elif pct <= 25:
        action = "keep"
    else:
        action = f"keep; {pct:.0f}% missing, usable only for the rest"
    rows.append({"check": f"Missing {col}", "result": missing, "action": action})

report = pd.DataFrame(rows)
print(report.to_string(index=False))

report.to_csv("../../Reports/day_89_cleaning_report.csv", index=False)

**What I found:** 1,756 layers; 0 duplicate rows; 0 duplicate `layer_id`; 0 impossible values; 144 unusually high carbon values (all from GH-GhaSP); `ph` missing in 14 layers, `clay` in 1,257 and `sand` in 1,256.

**Self-check:** missing + not missing = 1,756 for each column (14 + 1,742, 1,257 + 499, 1,256 + 500).

## 5. Two sentences for the reader
"Safe" needs a qualifier. Here is the corrected version:

- `ph` is usable (14 missing of 1,756). `clay` and `sand` are usable **only for the 499 and 500 layers that have values**, and none of those come from GH-GhaSP, so they cover about 28% of the layers.
- `orgc` is **not yet safe for pooled averages**: GH-GhaSP's carbon looks about 25 to 50 times higher than the other datasets, and the cause is not confirmed.

## 6. What I keep from today
- The numbers in a report come from code.
- An `action` starts with a verb and gives a why.
- "Safe" needs a qualifier: say how much of the data it covers.
- Do not claim more than the evidence shows ("cause not confirmed", not "a different scale").